# 01 — HMM on the IBL visual→motor triplet

**A per-region Poisson HMM: where the population is, and whether it runs a directed sequence.**
Regions `VISp` → `SCm` → `MRN` (primary visual cortex → motor-related superior colliculus →
midbrain reticular nucleus), the one axis triplet IBL co-records often enough to treat sessions
as replicates (5 sessions; this is the working one, `5c0c560e`).

This notebook is the **clean single-session pipeline** for the HMM strand — run it top-to-bottom
to see the results immediately. It reads the **unified preprocessing** (`src/data/preprocess.py`,
the same frozen tensors every model in the project eats), fits the HMM at the K chosen by **state
stability**, and reports the read-outs that survived scrutiny. To replicate across all 5 sessions
(both alignments, ARI curves, pooled stats, macro raster) run `scripts/run_hmm_5sessions.py` →
`results/hmm_5sessions/`.

> ### ⚠️ Read this before interpreting any late state
> The window is `[−0.3, +0.8] s` around **stimulus onset**, but the mice are fast: median
> reaction time **171 ms**, median feedback **427 ms**. So over `[0, +0.8] s` movement has begun
> on **92 %** of trials and the outcome has arrived on **83 %**. **In IBL the choice *is* the
> wheel-turn direction**, so "decision" and "movement" are not separable here. Every state after
> ≈ +0.2 s is entangled with movement and reward, and no such state may be called a "decision"
> state without the **`firstMovement`-aligned control** (built in `01_preprocessing`; run through
> the HMM for all 5 sessions by `scripts/run_hmm_5sessions.py --align both`). See
> `docs/worklog/2026-07-14_preprocessing-unification.md` §1.

Method notes: [`HMM_maths_and_readouts.md`](../docs/knowledge/HMM_maths_and_readouts.md),
[`choosing_K_for_the_HMM.md`](../docs/knowledge/choosing_K_for_the_HMM.md),
[`ARI_and_state_stability.md`](../docs/knowledge/ARI_and_state_stability.md),
[`best_K_per_region.md`](../docs/knowledge/best_K_per_region.md).

Colours: **Okabe-Ito** (red-green-colour-blind safe), **cividis** for unsigned heatmaps,
**RdBu_r** for signed; coloured lines carry a black outline so yellow stays legible.

## Part 0 — Setup

`K_PER_REGION` is **not** chosen here — it is imported from the stability analysis
([`best_K_per_region.md`](../docs/knowledge/best_K_per_region.md), FINAL 2026-07-14). Part 3 can
re-derive it (slow, optional); Part 4 onward simply uses it.

In [ ]:
import sys, time, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.preprocess import load_processed, preprocess_session, PreprocessConfig, save_processed
from src.data.build_matrices import firing_rate_table
from src.dynamics.hmm import (
    fit_poisson_hmm,                 # fit + baseline-biased startprob + time-ordering of states
    stability_scan,                  # Part 3 (optional): choose K by REPRODUCIBILITY, not likelihood
    net_flux,                        # N - N^T, the signed transition-flux matrix
    forward_fraction_split_half,     # directedness with the ordering derived on HELD-OUT trials
    switch_rate,                     # state changes / trial  (a FREQUENCY, not a direction)
    baseline_states,                 # which states ARE the pre-stimulus baseline (a set, from data)
    baseline_exit_latency_robust,    # when the population leaves baseline (works in every region)
)
from src.viz.style import set_style, state_colors, region_color, savefig, stroke, SEQUENTIAL, DIVERGING

set_style()

# ---- the ONE session, and the frozen preprocessing config --------------------------------
SESSION_EID = "5c0c560e-9e1f-45e9-b66e-e4ee7855be84"
REGIONS = ["VISp", "SCm", "MRN"]
SEED = 0

# K comes from the HMM stability analysis (docs/knowledge/best_K_per_region.md), NOT from this
# notebook. Only SCm K=4 is a genuine interior ARI peak; VISp/MRN K=3 are the smallest
# non-trivial K on a declining curve (see Part 3 and the doc's caveats).
K_PER_REGION = {"VISp": 3, "SCm": 4, "MRN": 3}

N_ITER = 50
rng = np.random.default_rng(SEED)
print("regions:", REGIONS, "| K per region:", K_PER_REGION)

## Part 1 — Load the unified tensors

Every model in the project now eats the **same** tensors, built once by
`src/data/preprocess.py` and cached in `data/processed/*.npz` with the config serialised
*inside* the file (so a tensor can never be silently mis-described later). This notebook no
longer builds its own — that was the source of four notebooks quietly disagreeing about "the
data" (different windows, bins, unit sets).

**Frozen config:** align `stimOn`, `pre 0.3 s` (a strict subset of IBL's enforced quiescence
floor), `post 0.8 s`, **40 ms bins**, units **≥ 1 Hz**, `qc == 1.0`. → VISp 19 / SCm 44 / MRN 54
units, 843 trials, 28 bins.

If the `.npz` is not present (fresh machine), the cell builds it from the raw session and
caches it — identical tensors either way.

In [ ]:
PROC = ROOT / "data" / "processed"
STIM_NPZ = PROC / f"{SESSION_EID[:8]}__stimOn_pre300_post800_bin40_rate1.npz"

if STIM_NPZ.exists():
    data = load_processed(STIM_NPZ)
    print(f"loaded {STIM_NPZ.name}")
else:
    # Fallback: build from the raw session (loads ~16M spikes; the only slow/heavy step).
    from src.data.access import get_one, sessions_table, load_trials, load_units
    from iblatlas.regions import BrainRegions
    import yaml
    params = yaml.safe_load((ROOT / "config" / "params.yaml").read_text())
    one = get_one(cache_dir=str((ROOT / params["data"]["one_cache_dir"]).resolve()))
    pids = sessions_table(one).query("eid == @SESSION_EID")["pid"].tolist()
    trials, mask = load_trials(one, SESSION_EID)
    br = BrainRegions()
    probe_loads = []
    for pid in pids:
        spikes, clusters = load_units(one, pid)
        clusters = clusters.copy()
        clusters["Beryl"] = br.id2acronym(clusters["atlas_id"].values, mapping="Beryl")
        probe_loads.append({"spikes": spikes, "clusters": clusters})
    cfg = PreprocessConfig()   # the frozen defaults (pre 0.3 s, 40 ms bins, >= 1 Hz)
    from src.data.preprocess import align_times, usable_mask
    keep = usable_mask(trials, mask, cfg.align_event)
    ev = align_times(trials, mask, cfg.align_event)
    data = preprocess_session(probe_loads, ev, cfg)
    tt = trials[mask][keep]      # trial covariates, in register with the tensors' axis 0
    data["trials"] = {"rt": (tt["firstMovement_times"] - tt["stimOn_times"]).to_numpy()}
    PROC.mkdir(parents=True, exist_ok=True)
    save_processed(data, STIM_NPZ)
    print(f"built + cached {STIM_NPZ.name}")

# ---- unpack into the names the rest of the notebook uses -----------------------------------
cfg = data["config"]
BIN_S = cfg["bin_size_s"]
bin_centers = data["bin_centers"]           # (28,) seconds relative to stimOn; <0 = pre-stimulus
baseline_mask = data["baseline_mask"]       # (28,) True for pre-stimulus bins
tensors = {r: data["regions"][r]["counts"] for r in REGIONS}   # each (trials, bins, units), raw int counts
rt = np.asarray(data["trials"]["rt"])       # reaction time per trial (s); a few NaN
n_trials = data["n_trials"]

print(f"\nconfig: align={cfg['align_event']}  window=[{-cfg['pre_s']}, +{cfg['post_s']}]s  "
      f"bin={int(BIN_S*1000)}ms  min_rate={cfg['min_rate_hz']}Hz")
print(f"{n_trials} trials, {len(bin_centers)} bins, baseline = {int(baseline_mask.sum())} pre-stimulus bins")
for r in REGIONS:
    print(f"  {r}: {tensors[r].shape}  (trials, bins, units)")
print(f"\nmedian RT = {np.nanmedian(rt)*1000:.0f} ms   "
      f"(so the window is mostly MOVEMENT — see the warning at the top)")

## Part 2 — A sanity look at the spikes

Not analysis — just confirming the tensors are what we think before modelling them.

- **Firing table**: IBL "good" units are *well-isolated*, not *active*; the `≥ 1 Hz` filter has
  already removed the near-silent tail (which otherwise pushes Poisson rates to zero and breaks
  cross-validation — see `choosing_K_for_the_HMM.md`).
- **Population PSTH** (mean rate ± SEM across trials) and **per-unit PSTH heatmap**
  (peak-normalised, sorted by peak latency). Look for a change at `t = 0`, and for the
  visual→motor latency ordering.

*(The pynapple cross-correlogram / autocorrelogram QC pass that used to live here needs raw
spike trains, which the processed tensors deliberately do not carry. It has moved out of this
findings pipeline; `src/viz/nap_plots.py` still provides it for exploration.)*

In [ ]:
rows = []
for region in REGIONS:
    counts = tensors[region]
    ft = firing_rate_table(counts, BIN_S)
    rows.append({
        "region": region, "n_units": counts.shape[2],
        "pop_mean_rate_hz": round(ft.mean_rate_hz.mean(), 2),
        "median_unit_rate_hz": round(ft.mean_rate_hz.median(), 2),
        "median_spikes_per_trial": round(ft.spikes_per_trial.median(), 2),
        "lambda_per_bin": round(counts.mean(), 3),   # mean spikes/bin -> the Poisson rate regime
    })
print("Population firing summary (per region):")
display(pd.DataFrame(rows))

fig, axes = plt.subplots(2, 3, figsize=(15, 6.5), gridspec_kw={"height_ratios": [1, 1.3]})
for j, region in enumerate(REGIONS):
    counts = tensors[region]; c = region_color(region); n_units = counts.shape[2]
    # population PSTH
    psth = counts.mean(axis=(0, 2)) / BIN_S
    sem = counts.mean(axis=2).std(axis=0) / np.sqrt(counts.shape[0]) / BIN_S
    axes[0, j].plot(bin_centers, psth, color=c, lw=1.8, path_effects=stroke())
    axes[0, j].fill_between(bin_centers, psth - sem, psth + sem, alpha=0.25, color=c, lw=0)
    axes[0, j].axvline(0, color="0.3", ls="--", lw=1)
    axes[0, j].set(title=f"{region}: population PSTH (n={n_units})", ylabel="rate (sp/s)")
    # per-unit PSTH heatmap, peak-normalised, sorted by peak latency
    up = counts.mean(axis=0).T / BIN_S
    norm = up / (up.max(axis=1, keepdims=True) + 1e-9)
    order = np.argsort(norm.argmax(axis=1))
    im = axes[1, j].imshow(norm[order], aspect="auto", cmap=SEQUENTIAL,
                           extent=[bin_centers[0], bin_centers[-1], n_units, 0])
    axes[1, j].axvline(0, color="w", ls="--", lw=1)
    axes[1, j].set(xlabel="time from stimOn (s)", ylabel="unit (sorted by peak)")
fig.colorbar(im, ax=axes[1, :], label="peak-normalised rate", fraction=0.03, pad=0.02)
fig.suptitle("Per-region activity (stimulus-aligned)", y=0.99)
savefig(fig, "phase2_activity", "region_activity", root=ROOT)
plt.show()

## Part 3 — How K was chosen (state stability), and why *not* likelihood

**This is an explanation, not a live selection — `K_PER_REGION` is already set in Part 0.**
Run the optional cell to reproduce the stability curves (~4–8 min/region); otherwise read on.

**Held-out likelihood does not work here, structurally.** A Poisson HMM assumes neurons are
*conditionally independent given the state*; real populations are correlated; and the model's
*only* way to express correlation is to add a state. So held-out log-likelihood climbs with K
forever, and its argmax lands on whichever end of the search range you look at — it gave **K=2**
(bottom, off a column of `-inf`) and then **K=10** (top) on the same data. Both are boundaries,
not answers. (The implementation is fine: `hmmlearn` matches `dynamax` to machine precision —
`scripts/validate_hmm_vs_dynamax.py`.)

**What we use instead — reproducibility.** The claim is *reproducible metastable states*, so
measure reproducibility: fit two HMMs on disjoint trial thirds **A** and **B**, decode both on a
never-seen third **C**, and score how much the two independent fits agree, with the
**Adjusted Rand Index** (permutation-invariant, chance-corrected, comparable across K). Unlike
likelihood it has a *peak*, and it can say *"no reproducible states here"*. Full argument:
[`ARI_and_state_stability.md`](../docs/knowledge/ARI_and_state_stability.md).

**Result (843 trials, 5 repeats × 2 EM restarts, [`best_K_per_region.md`](../docs/knowledge/best_K_per_region.md)):**

| region | K | ARI | reading |
|---|---:|---|---|
| VISp | **3** | 0.883 ± 0.019 | *not a peak* — ARI declines monotonically from K=2; 3 is the smallest non-trivial K |
| SCm | **4** | **0.920 ± 0.031** | a genuine interior peak — the one solid state-count result in the project |
| MRN | **3** | 0.882 ± 0.052 | tied with the trivial split (0.8819 vs 0.8802); only replication can settle it |

⚠️ **Do not read the raw peak (K=2 everywhere, ARI ≈ 0.98).** A two-state model only has to find
the baseline-vs-evoked step — one huge event both fits recover every time. *Stability is not
correctness.* The table reports the best **non-trivial** (K ≥ 3) structure.

In [ ]:
# OPTIONAL: reproduce the stability curves. Slow (~4-8 min/region). Off by default.
RUN_STABILITY = False
STAB_KS = [2, 3, 4, 5, 6, 7, 8]
STAB_REPEATS = 3       # 5 in the definitive run (best_K_per_region.md); 3 here for speed
STAB_RESTARTS = 2      # EM restarts per fit — WITHOUT this the ARI (and K) is biased DOWN
STAB_ITER = 25

if not RUN_STABILITY:
    print("RUN_STABILITY = False — using K from best_K_per_region.md:", K_PER_REGION)
    print("Set RUN_STABILITY = True to re-derive the ARI-vs-K curves on this machine.")
else:
    fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.4))
    for region in REGIONS:
        counts = tensors[region]
        t0 = time.time()
        st = stability_scan(counts, STAB_KS, n_repeats=STAB_REPEATS, n_iter=STAB_ITER,
                            seed=SEED, n_restarts=STAB_RESTARTS)
        K = st["n_states"].to_numpy(); ari = st["ari_mean"].to_numpy()
        nontrivial = K >= 3
        peak = int(K[nontrivial][np.argmax(ari[nontrivial])])
        c = region_color(region)
        ax[0].errorbar(K, ari, yerr=st["ari_std"], marker="o", ms=5, capsize=3, lw=1.8,
                       color=c, label=f"{region} (best K≥3: {peak})", path_effects=stroke())
        ax[0].scatter([peak], [ari[list(K).index(peak)]], s=170, facecolors="none",
                      edgecolors=c, linewidths=2.2, zorder=5)
        ax[1].plot(K, st["matched_acc_mean"], marker="o", ms=5, lw=1.8, color=c,
                   label=region, path_effects=stroke())
        print(f"{region}: best non-trivial K = {peak}  (ARI={ari[nontrivial].max():.3f})  [{time.time()-t0:.0f}s]")
    ax[0].axvspan(1.6, 2.4, color="0.9", zorder=0)
    ax[0].axhline(0, color="black", lw=1, ls=":")
    ax[0].set(xlabel="K", ylabel="Adjusted Rand Index (0 = chance)", xticks=STAB_KS,
              title="State stability — circled = best NON-TRIVIAL K\n(K=2 grey = the trivial baseline/evoked split)")
    ax[0].legend(frameon=False, fontsize=8)
    ax[1].set(xlabel="K", ylabel="matched accuracy (Hungarian)", xticks=STAB_KS,
              title="Matched agreement — NOT chance-corrected\n(falls with K for free; read with the ARI)")
    ax[1].legend(frameon=False, fontsize=8)
    fig.tight_layout(); savefig(fig, "phase2_cv", "state_stability", root=ROOT); plt.show()

## Part 4 — Fit the HMM at the chosen K, per region

`fit_poisson_hmm` (see [`HMM_maths_and_readouts.md`](../docs/knowledge/HMM_maths_and_readouts.md)):
fits Poisson emissions on **raw counts** (never smoothed); **biases the initial state** to the
measured pre-stimulus baseline occupancy so the post-stimulus path reads as a *departure from
baseline*; and **orders states by occupancy centre-of-mass** so that "later state" — and hence
"forward" — is well defined.

**Three panels per region.** *Left* — Viterbi state per (trial, time), trials sorted by reaction
time, white curve = first-movement time. *Middle* — state occupancy P(state | time), baseline
shaded. *Right* — net transition flux `N − Nᵀ` (red above the diagonal = a net forward step,
blue = a return step); its one-number summary is `forward_fraction`.

In [ ]:
results = {}
t0 = time.time()
for region in REGIONS:
    counts = tensors[region]
    K = K_PER_REGION[region]
    res = fit_poisson_hmm(counts, K, baseline_mask, n_iter=N_ITER, seed=SEED, bias_startprob=True)
    res.bin_centers = bin_centers
    results[region] = res
    print(f"{region}: K={K}  converged={res.model.monitor_.converged}  "
          f"forward_fraction={res.forward_fraction:.3f}  switch_rate={switch_rate(res.states):.2f}/trial")
print(f"[{time.time()-t0:.1f}s]")

In [ ]:
def plot_hmm(region, res):
    """State raster (trials sorted by RT), state occupancy, and net-flux matrix."""
    bc, K = res.bin_centers, res.occupancy.shape[0]
    cols = state_colors(K)
    order = np.argsort(rt)                       # NaN RTs sort to the end
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))

    im0 = ax[0].imshow(res.states[order], aspect="auto", interpolation="nearest",
                       extent=[bc[0], bc[-1], len(order), 0], cmap=SEQUENTIAL, vmin=0, vmax=K - 1)
    ax[0].plot(rt[order], np.arange(len(order)) + 0.5, color="w", lw=1.2, label="first move",
               path_effects=stroke(2.2, "black"))
    ax[0].axvline(0, color="w", ls="--", lw=1)
    ax[0].set(xlabel="time from stimOn (s)", ylabel="trial (sorted by RT)",
              xlim=(bc[0], bc[-1]), title=f"{region}: HMM state (K={K})")
    ax[0].legend(frameon=False, loc="lower right", fontsize=8, labelcolor="w")
    fig.colorbar(im0, ax=ax[0], label="state index", ticks=range(K))

    bstates = baseline_states(res.occupancy, bc)     # which states ARE baseline (a set)
    for k in range(K):
        lbl = f"state {k}" + ("  (baseline)" if k in bstates else "")
        ax[1].plot(bc, res.occupancy[k], color=cols[k], label=lbl, path_effects=stroke())
    ax[1].axvspan(bc[0], 0, color="0.9")
    ax[1].axvline(0, color="0.3", ls="--", lw=1)
    ax[1].set(xlabel="time from stimOn (s)", ylabel="P(state)", title="state occupancy")
    ax[1].legend(frameon=False, fontsize=8, ncol=2)

    flux = net_flux(res.trans_counts)
    vmax = np.abs(flux).max() or 1
    im = ax[2].imshow(flux, cmap=DIVERGING, vmin=-vmax, vmax=vmax)
    for i in range(K):
        for j in range(K):
            if i != j:
                ax[2].text(j, i, f"{flux[i, j]:+d}", ha="center", va="center", fontsize=8)
    ax[2].set(xlabel="to state", ylabel="from state", xticks=range(K), yticks=range(K),
              title=f"net flux N−Nᵀ  (fwd frac={res.forward_fraction:.2f})")
    fig.colorbar(im, ax=ax[2], label="net i→j transitions")
    fig.tight_layout()
    savefig(fig, "phase2_hmm", f"{region}_hmm", root=ROOT)
    return fig

for region in REGIONS:
    plot_hmm(region, results[region])
plt.show()

## Part 5 — The read-outs that survive

Three numbers per region, each computed **honestly** (the caveats below are the whole point):

**`forward_fraction` — directedness.** Share of transitions that run to a *later*-ordered state.
0.5 = symmetric mixing, →1 = a clean forward chain. But the ordering is derived from occupancy,
so ordering-then-scoring-the-same-trials is nearly circular. **`forward_fraction_split_half`**
derives the ordering on *held-out* trials and is the number to quote.

**`switch_rate` — how often the population changes state at all.** This is a *frequency*, not a
direction, and the two are constantly conflated. A region can switch constantly with no preferred
direction (high switch_rate, forward_fraction ≈ 0.5) — which is exactly what VISp does.

**`baseline_exit_latency_robust` — when the population leaves its baseline set.** Threshold is
relative to baseline's *own* pre-stimulus level (the old absolute-0.5 rule returned `NaN` for 2
of 3 regions). ⚠️ Interpret against the movement window: SCm's exit sits near the median RT
(171 ms) and MRN's near median feedback (427 ms), so the "cascade" may be each area locking to
the task event it already locks to — not an internal decision sequence.

In [ ]:
readout = []
for region in REGIONS:
    counts = tensors[region]
    res = results[region]
    K = K_PER_REGION[region]

    sh = forward_fraction_split_half(counts, K, baseline_mask, n_repeats=5, seed=SEED, n_iter=N_ITER)
    lat = baseline_exit_latency_robust(res.occupancy, res.bin_centers)

    readout.append({
        "region": region, "K": K, "n_units": counts.shape[2],
        "forward_fraction": round(res.forward_fraction, 3),          # pooled (ordering on same trials)
        "fwd_frac_split_half": round(sh["forward_fraction_split_half"], 3),
        "fwd_frac_sh_std": round(sh["std"], 3),
        "switch_rate": round(switch_rate(res.states), 2),
        "baseline_exit_ms": None if np.isnan(lat["latency_s"]) else round(lat["latency_s"] * 1000),
    })

readout_df = pd.DataFrame(readout)
display(readout_df)
(ROOT / "results" / "hmm").mkdir(parents=True, exist_ok=True)
readout_df.to_csv(ROOT / "results" / "hmm" / "hmm_readouts_5c0c560e.csv", index=False)
print("saved -> results/hmm/hmm_readouts_5c0c560e.csv")

print("\nHow to read this table:")
print("  forward_fraction vs split_half : if they agree, directedness is real. MRN is the one to")
print("                                   watch — the split-half test drops it (ordering circularity).")
print("  switch_rate                    : VISp switches about as often as MRN, but UNDIRECTED — the")
print("                                   forward_fraction/switch_rate dissociation, measured.")
print("  baseline_exit_ms               : the visual->motor cascade, BUT confounded with RT/feedback.")

## Part 6 — What the HMM shows, what was retracted, and the limits

**What survives.**
- **Motor midbrain traverses its states far more directedly than visual cortex.** `VISp ≈ 0.59`
  (at chance — undirected) vs `SCm ≈ 0.83` and `MRN` (split-half) `≈ 0.67 ± 0.10`. The *ordering*
  is stable across every bin size and K we have tried; it is the most robust HMM result.
- **VISp is undirected, not static.** It switches state about as often as MRN — it simply switches
  with no preferred direction. (`forward_fraction` is an asymmetry; `switch_rate` is a frequency.)
- **SCm K=4 is the one solid state-count result** — a genuine, tight ARI peak (0.920 ± 0.031).

**Retracted.**
- **There is *no* visual→motor gradient in the number of states.** An earlier pass claimed
  VISp 2 < MRN 3 < SCm 4; with EM restarts the counts are **3 / 4 / 3** — VISp and MRN are equal.
  The honest synthesis: *the areas hold a similar number of states, but the motor areas traverse
  them more directedly* (shared alphabet, area-specific sequence).
- **MRN's directedness is weaker than first reported.** The split-half test dropped it from ~0.79
  to ~0.67 ± 0.10 — part of the apparent directedness was ordering-then-testing-the-ordering.

**Limits (stated plainly).**
1. ⚠️ **The window is mostly movement and reward** (top-of-notebook warning). The directedness
   and the latency cascade both have an unglamorous null — *they reflect movement, not decision* —
   that only the `firstMovement`-aligned control can exclude. That control is now **run** for all 5
   sessions by `scripts/run_hmm_5sessions.py --align both` (compare the `stimOn` vs `firstMovement`
   figures in `results/figures/hmm_5sessions/`).
2. **One session here.** The way to strengthen any of this is to **replicate across the 5 triplet
   sessions and pool the statistics** ("K=3 in n/5 sessions, forward_fraction 0.83 ± …") — not to
   build a pseudo-population, which destroys the single-trial joint activity these models consume.
   **`scripts/run_hmm_5sessions.py`** runs exactly this pipeline over every session at the current
   window, with the ARI-vs-K curves and the cross-session macro raster, into `results/hmm_5sessions/`.
3. **VISp's K=3 is not an ARI peak, and MRN's is tied with the trivial split** — both need
   replication, not more repeats, to settle. (In the 5-session scan SCm/MRN land on K=3 defensibly;
   VISp has no cross-session consensus — consistent with its weak, declining ARI curve.)

## Part 7 — Do the states track the *task phase*? (IBL now; monkey working-memory dataset TODO)

**The question.** If the states are real task variables, the decoded state should line up with the
*phases of the trial*, not just with the wall clock. For IBL we can define three phases from events
already in the tensor: **pre-stim** (`t<0`), **stim→move** (`0 ≤ t < RT`), **post-move** (`t ≥ RT`,
`RT = firstMovement − stimOn`). For **colleague D.'s monkey working-memory dataset** (`notebooks/07`,
DANDI `000620`, areas **FEF** + **DMFC**) the same idea has *more* phases — fixation / sample /
delay / go / response — and is the natural next application (see the TODO below).

**⚠️ The null is the whole analysis.** Both the state and the phase are strong functions of absolute
time, so raw mutual information between them is high *for free* even if the state ignores the task.
The honest test asks whether the state tracks the **trial-specific** timing (chiefly the variable
movement latency RT). So we reference the observed NMI to a **trial-permutation null** that shuffles
each trial's RT across trials — preserving the state sequences and the absolute-time phase structure,
destroying only the per-trial coupling. Alignment *above* this null is what the clock alone cannot
explain. (Same discipline as `order_consistency`'s null: compare to the shuffle, never to 0.5.)

**Implemented and run** — `src/dynamics/hmm.py::{phase_labels, state_phase_alignment,
state_locked_occupancy}` and `scripts/run_hmm_state_phase.py` (→ `results/hmm/state_phase.csv`,
`results/figures/hmm_state_phase/`). Per-session-then-pool over all 5 triplet sessions (K = 3 all
regions, the 5-session common K):

| region | beats null | mean z | Cramér's V |
|---|:--:|--:|--:|
| VISp | **3/5** | 12 ± 20 | 0.15 ± 0.10 |
| SCm  | **5/5** | 16 ± 13 | 0.37 ± 0.16 |
| MRN  | **5/5** | 13 ± 13 | 0.34 ± 0.17 |

**Reading.** The motor areas' states track the trial's phase above the clock-null in **every**
session; VISp does so weakly and only in 3/5 — the *same* motor-directed / visual-undirected
dissociation Part 6 found, seen through a second lens. **But** the movement-locked occupancy panel
makes plain that most of this alignment *is* the baseline→evoked switch snapping to movement onset —
so it inherits the load-bearing caveat at the top of the notebook: it beats a clock-only account, it
does **not** separate movement from decision. The `firstMovement`-aligned tensors are the control.

**TODO — monkey working-memory dataset (FEF + DMFC, DANDI 000620).**
- **Design (answering the "merge sessions, keep areas separate?" question):** fit **one Poisson HMM
  per session × area** and **pool the statistics** — the same replicate-don't-concatenate rule we use
  for IBL. FEF and DMFC are recorded *simultaneously* within a session (identical trial counts), so
  they share single-trial activity and each is its own population; but neurons differ across the 14
  usable sessions, so there is **no shared population to merge** — merging would build a
  pseudo-population and destroy the single-trial joint activity the HMM consumes. "Keep the areas
  separate" ✓; "merge the sessions" → pool the read-outs (`forward_fraction`, K, phase-alignment z),
  not the neurons. 14 sessions is a *stronger* replication substrate than IBL's 5.
- **Blocked on:** the task-event timestamps (fixation/sample/delay/go/response) — the current pickle
  carries only per-(trial, neuron) spike times, no epoch marks. Once obtained, feed them to
  `state_phase_alignment` as `landmark_times` and reuse this whole section unchanged.
- **Then compare:** does the FEF→DMFC pair show the visual-cortex→motor style directedness/phase-
  alignment gradient the IBL triplet does, on a task with a real delay period?
